In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI

token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"
model = "gpt-4o-mini"

openai_client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

In [22]:
def llm(prompt):
    # CORRECCIÓN: Usar el endpoint clásico .chat.completions.create
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}  # El formato correcto para pasar el texto
        ]
    )
    # CORRECCIÓN: La respuesta de chat completions se extrae así
    return response.choices[0].message.content

In [73]:
print(llm("Hey, what's up?"))

Hello! I'm here to help you with any questions or topics you want to discuss. What’s on your mind?


In [64]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

It depends on the specific course you’re referring to. Many courses offer rolling admissions or have flexible start dates, while others might have fixed enrollment periods. I recommend checking the course's official website or contacting the course provider to get detailed information about enrollment options and deadlines.


In [77]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [78]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [79]:
answer = llm(prompt)
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [9]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [11]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [12]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [13]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [14]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [15]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [16]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [17]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [18]:
search_results = search(question)

In [19]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [20]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [21]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [22]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [23]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [24]:
response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "developer", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt}  # El formato correcto para pasar el texto
        ]
    )

In [25]:
response.choices[0].message.content

'Yes, you can still join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [26]:
response.usage

CompletionUsage(completion_tokens=34, prompt_tokens=533, total_tokens=567, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0), latency_checkpoint={'engine_tbt_ms': 8, 'engine_ttft_ms': 36, 'engine_ttlt_ms': 321, 'pre_inference_ms': 113, 'service_tbt_ms': 12, 'service_ttft_ms': 1123, 'service_ttlt_ms': 1520, 'total_duration_ms': 1411, 'user_visible_ttft_ms': 1010})

In [27]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.prompt_tokens * input_price +
    response.usage.completion_tokens * output_price
)

cost

0.00055275

In [31]:
def llm(instructions, user_prompt, model="gpt-4o-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history
    )

    return response.choices[0].message.content

In [32]:
def rag(query, model="gpt-4o-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [33]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [34]:
rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a "live" cohort. Certificates are not awarded for completing the course in a self-paced mode. Additionally, you must pass the Capstone project, as it is a requirement for certification. It’s also important to note that you need to peer-review three capstone projects after submitting your own, which can only be done while the course is running. If you missed the first homework, it is not mandatory for obtaining the certificate, but passing the Capstone project is essential.'

# Using .py Modules

In [35]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

token = os.getenv("GITHUB_TOKEN")
endpoint = "https://models.github.ai/inference"

openai_client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [36]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [39]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course after it has started. However, if you want to receive a certificate, you need to submit your project while the submissions are still being accepted.'

In [38]:
assistant.rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a "live" cohort. Certificates are not awarded for self-paced courses because peer-reviewing capstone projects is a requirement, and this can only be done while the course is running. Additionally, you must pass the Capstone project to obtain your certificate, although completing homework is not mandatory. If you wish to avoid having "Lucid Elbakyan" on your certificate, make sure to enter your official name in the course profile settings.'